In [55]:
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np
import torch

# Add path ../../01_agentic_rag/ to sys.path
import sys
sys.path.append("../../01_agentic_rag/")
from ingestion import load_faq_data, build_index
from rag_helper import RAGBase

In [ ]:
print("torch:", torch.__version__)
print("torch cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

torch: 2.12.0+cu130
torch cuda build: 13.0
cuda available: False


In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")


/home/daniel/code/dosorio79/llm-zoomcamp/Lessons/.venv/lib/python3.14/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

When vector embeddings are normalized to a length of 1, their dot product becomes exactly equal to their cosine similarity. 

### Why This Happens

* **The Formula**: Cosine similarity is calculated as:
  $$\text{Cosine Similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$
* **The Simplification**: If vectors $A$ and $B$ are normalized, their magnitudes ($\|A\|$ and $\|B\|$) both equal $1$. 
* **The Result**: The denominator becomes $1 \times 1$, reducing the equation to just the dot product ($A \cdot B$).

### Why Use Normalized Dot Product?

* **Speed**: Dot product requires fewer calculations. It eliminates square roots and divisions during search queries.
* **Hardware Efficiency**: Modern CPUs and GPUs process dot products extremely fast via highly optimized matrix multiplication.
* **Consistency**: It bounds the similarity score strictly between $-1.0$ and $+1.0$.

### Implementation Tip

Most vector databases (like Pinecone, Milvus, or Qdrant) run much faster when you configure them for **Dot Product** distance while feeding them pre-normalized embeddings from models like OpenAI's `text-embedding-3` or Cohere's v3.

In [5]:
v1.dot(dv)

np.float32(0.323324)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v2.dot(dv)

np.float32(0.019730432)

# Load and embed the documents

In [ ]:
# Load the documents
documents = load_faq_data()

Loaded 6 courses
Fetching: https://datatalks.club/faq//json/data-engineering-zoomcamp.json
Added 404 documents from Data Engineering Zoomcamp (course: data-engineering-zoomcamp)
Fetching: https://datatalks.club/faq//json/stock-markets-analytics-zoomcamp.json
Added 93 documents from Stock Markets Analytics Zoomcamp (course: stock-markets-analytics-zoomcamp)
Fetching: https://datatalks.club/faq//json/ai-dev-tools-zoomcamp.json
Added 41 documents from AI Dev Tools Zoomcamp (course: ai-dev-tools-zoomcamp)
Fetching: https://datatalks.club/faq//json/llm-zoomcamp.json
Added 84 documents from LLM Zoomcamp (course: llm-zoomcamp)
Fetching: https://datatalks.club/faq//json/mlops-zoomcamp.json
Added 255 documents from MLOps Zoomcamp (course: mlops-zoomcamp)
Fetching: https://datatalks.club/faq//json/machine-learning-zoomcamp.json
Added 472 documents from ML Zoomcamp (course: machine-learning-zoomcamp)
Total documents loaded: 1349


In [ ]:
# Create a list of texts from the documents
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [16]:
len(texts)

1349

In [ ]:
# Create embeddings for all the texts in batches
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1349

In [14]:
# Convert the list of vectors to a NumPy array
X = np.array(vectors)

In [ ]:
# 1349 texts, 384 dimensions from the embedding model
X.shape

(1349, 384)

# scoring documents by cosine similarity

In [18]:
# Create a query and encode it
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [19]:
# score the query against all the vectors
scores = X.dot(v_query)

In [20]:
scores

array([ 0.48740578,  0.20991933,  0.7629412 , ..., -0.08637966,
        0.03759794, -0.03037041], shape=(1349,), dtype=float32)

In [21]:
# similar list comprehension  logic slower
scores = [X[i].dot(v_query) for i in range(X.shape[0])]

In [22]:
scores

[np.float32(0.48740578),
 np.float32(0.20991933),
 np.float32(0.7629412),
 np.float32(0.44378537),
 np.float32(0.26084),
 np.float32(0.48665166),
 np.float32(0.30061555),
 np.float32(0.56009996),
 np.float32(0.45960492),
 np.float32(0.25628173),
 np.float32(0.33153278),
 np.float32(0.27318528),
 np.float32(0.2769164),
 np.float32(0.34123006),
 np.float32(0.4600717),
 np.float32(0.26240283),
 np.float32(0.3920009),
 np.float32(0.108541705),
 np.float32(0.2756731),
 np.float32(0.16646813),
 np.float32(0.31437927),
 np.float32(0.006685457),
 np.float32(0.1120503),
 np.float32(0.21905686),
 np.float32(0.3400085),
 np.float32(0.23571303),
 np.float32(0.32714847),
 np.float32(0.15088364),
 np.float32(0.1656328),
 np.float32(0.0554502),
 np.float32(0.23541196),
 np.float32(0.085330196),
 np.float32(-0.0030899234),
 np.float32(-0.042598397),
 np.float32(-0.06027717),
 np.float32(0.006491076),
 np.float32(0.034277506),
 np.float32(-0.049589735),
 np.float32(-0.0006708354),
 np.float32(-0.017013

In [27]:
# Highest scoring document
best_idx = np.argmax(scores)
best_doc = documents[best_idx]
display(f"best_idx: {best_idx}, score: {scores[best_idx]:.2f}", best_doc)


'best_idx: 2, score: 0.76'

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [34]:
# Get the top 5 highest scoring documents from largest to smallest
top5_idx = np.argsort(scores)[-5:][::-1]

In [35]:
top5

array([  2, 624, 906, 538,   7])

In [38]:
# Get the top 5 highest scoring documents
top5_docs = [documents[i] for i in top5_idx]

In [39]:
top5_docs

[{'id': '3f1424af17',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questi

# Vector search with minsearch

In [47]:
from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [48]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [49]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [ ]:
# With filter for course llm-zoomcamp
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# Vector RAG

### RAG with text indexing and retrieval

In [53]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [56]:
documents = load_faq_data()
index = build_index(documents)

Loaded 6 courses
Fetching: https://datatalks.club/faq//json/data-engineering-zoomcamp.json
Added 404 documents from Data Engineering Zoomcamp (course: data-engineering-zoomcamp)
Fetching: https://datatalks.club/faq//json/stock-markets-analytics-zoomcamp.json
Added 93 documents from Stock Markets Analytics Zoomcamp (course: stock-markets-analytics-zoomcamp)
Fetching: https://datatalks.club/faq//json/ai-dev-tools-zoomcamp.json
Added 41 documents from AI Dev Tools Zoomcamp (course: ai-dev-tools-zoomcamp)
Fetching: https://datatalks.club/faq//json/llm-zoomcamp.json
Added 84 documents from LLM Zoomcamp (course: llm-zoomcamp)
Fetching: https://datatalks.club/faq//json/mlops-zoomcamp.json
Added 255 documents from MLOps Zoomcamp (course: mlops-zoomcamp)
Fetching: https://datatalks.club/faq//json/machine-learning-zoomcamp.json
Added 472 documents from ML Zoomcamp (course: machine-learning-zoomcamp)
Total documents loaded: 1349
Building index with 1349 documents
Text fields: ['question', 'sectio

In [57]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

[INIT] Initializing RAGBase
[INIT] RAGBase initialized with model: gpt-5.4-mini


In [58]:
query = "I just found out about the program, can I still sign up?"

In [59]:
assistant.rag(query)

[SEARCH] Query: I just found out about the program, can I still sign up?, Course: llm-zoomcamp, Num Results: 5
[SEARCH] Found 5 results
[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 1925


'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

### RAG with vector search and retrieval

In [60]:
# Create a new class that inherits from RAGBase and implements the retrieve method using minsearch

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )



In [61]:
# instantiate the RAGVector class with the minsearch index and the embedding model
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

[INIT] Initializing RAGBase
[INIT] RAGBase initialized with model: gpt-5.4-mini


In [62]:
vector_assistant.rag(query)

[BUILD_CONTEXT] Building context from 5 search results
[BUILD_CONTEXT] Context built with length: 1880


'Yes — you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

# Vector search with sqlite and pgvector